In [1]:
pip install pandas langdetect

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

In [2]:
import requests
import json
import time
import datetime
import os                     
from dotenv import load_dotenv


load_dotenv()


hashtags = [
    "france", "français", "française", "francophone", "paris",
    "parisien", "parisienne", "lyon", "marseille", "bordeaux",
    "ptp", "pourtoi", "pourtapage", "pourvouspage", "amusant",
    "musique", "mode", "suivre", "TikTok4amusant", "comédie",
    "bêtisier", "tendance", "blague", "francetiktok", "maroc",
    "marocain", "marocaine", "canadien", "canadienne"
]

target_accounts = []

url = "https://tiktok-scraper7.p.rapidapi.com/feed/search"

my_api_key = os.getenv("RAPIDAPI_KEY")

headers = {
    "x-rapidapi-host": "tiktok-scraper7.p.rapidapi.com",
    "x-rapidapi-key": my_api_key  
}

TARGET_FRENCH_COUNT = 1000
seen_video_ids = set()
total_scraped = 0
output_file = 'french_videos_raw.jsonl'

search_terms = hashtags + target_accounts

with open(output_file, 'a', encoding='utf-8') as f:

    for term in search_terms:
        if total_scraped >= TARGET_FRENCH_COUNT:
            break

        current_cursor = "0"
        has_more = True
        retry_count = 0

        while has_more and total_scraped < TARGET_FRENCH_COUNT:
            querystring = {
                "keywords": term,
                "region": "fr",
                "count": "30",
                "cursor": current_cursor,
                "publish_time": "0",
                "sort_type": "0"
            }

            try:
                response = requests.get(url, headers=headers, params=querystring)


                if response.status_code == 429:
                    retry_count += 1
                    if retry_count > 3:
                        print(f"Rate limit exceeded for '{term}'. Moving to next keyword...")
                        break
                    time.sleep(15)
                    continue

                elif response.status_code != 200:
                    print(f"API Error: {response.status_code} for term '{term}'")
                    break

                retry_count = 0
                data = response.json()


                videos_list = []
                meta_data = {}

                if isinstance(data, dict):
                    inner_data = data.get("data", {})


                    if isinstance(inner_data, dict):
                        videos_list = inner_data.get("videos", [])
                        meta_data = inner_data

                    elif isinstance(inner_data, list):
                        videos_list = inner_data


                    if not videos_list and isinstance(data.get("videos"), list):
                        videos_list = data.get("videos")

                if not isinstance(videos_list, list):
                    videos_list = []
                # --------------------------------

                if not videos_list:
                    print(f"No more results for: {term}")
                    break

                for video in videos_list:

                    if not isinstance(video, dict):
                        continue

                    raw_id = video.get("video_id") or video.get("aweme_id") or video.get("id")
                    video_id = str(raw_id) if raw_id is not None else ""

                    if video_id and video_id not in seen_video_ids:
                        seen_video_ids.add(video_id)


                        video['search_term_used'] = term

                        f.write(json.dumps(video, ensure_ascii=False) + '\n')
                        total_scraped += 1

                        if total_scraped >= TARGET_FRENCH_COUNT:
                            break

                if isinstance(meta_data, dict):
                    current_cursor = str(meta_data.get("cursor", int(current_cursor) + 30))
                    has_more = meta_data.get("hasMore", False) or meta_data.get("has_more", False)
                else:
                    current_cursor = str(int(current_cursor) + 30)
                    has_more = False

            except Exception as e:
                print(f"Error encountered: {e}")
                break


            time.sleep(4)


In [3]:
print(f"Total unique videos saved: {total_scraped}")
print(f"All raw metadata saved to '{output_file}'")

Total unique videos saved: 1000
All raw metadata saved to 'french_videos_raw.jsonl'


In [4]:
df = pd.read_json('french_videos_raw.jsonl', lines=True)

In [8]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

In [9]:
DetectorFactory.seed = 0

In [7]:



try:
    df_raw = pd.read_json('french_videos_raw.jsonl', lines=True)

except FileNotFoundError:

    df_raw = pd.DataFrame()

if not df_raw.empty:
    cleaned_videos = []

    for index, row in df_raw.iterrows():

        text = row.get("title") or row.get("desc") or ""


        if isinstance(text, (dict, list)):
            text = str(text)
        else:
            text = str(text).strip()


        if text:
            try:
                lang = detect(text)
                if lang == 'fr':

                    raw_time = row.get("create_time")
                    formatted_date = ""
                    if raw_time and pd.notna(raw_time):
                        try:
                            formatted_date = datetime.datetime.fromtimestamp(int(raw_time)).strftime('%Y-%m-%d %H:%M:%S')
                        except (ValueError, TypeError):
                            pass


                    stats = row.get("statistics") if isinstance(row.get("statistics"), dict) else {}
                    author = row.get("author") if isinstance(row.get("author"), dict) else {}

                    video_id = row.get("video_id") or row.get("aweme_id") or row.get("id")


                    video_info = {
                        "search_hashtag": row.get("search_term_used", ""),
                        "video_id": str(video_id) if pd.notna(video_id) else "",
                        "real_video_url": f"https://www.tiktok.com/@placeholder/video/{video_id}" if pd.notna(video_id) else "",
                        "create_time": raw_time if pd.notna(raw_time) else "",
                        "publish_date": formatted_date,
                        "clean_text": text,
                        "author_name": author.get("nickname") or author.get("unique_id") or "",
                        "play_count": stats.get("play_count", 0),
                        "digg_count": stats.get("digg_count", 0)
                    }
                    cleaned_videos.append(video_info)

            except LangDetectException:

                pass


    df_cleaned = pd.DataFrame(cleaned_videos)


    df_cleaned.drop_duplicates(subset=['video_id'], keep='first', inplace=True)

    output_csv = 'final_pure_french_1000_for_apify.csv'
    df_cleaned.to_csv(output_csv, index=False, encoding='utf-8-sig')


    print(f"French videos: {len(df_cleaned)}")
    print(f"Saved as: '{output_csv}'")


French videos: 723
Saved as: 'final_pure_french_1000_for_apify.csv'


In [1]:
import requests
import json
import time
import datetime
import os                     
from dotenv import load_dotenv


load_dotenv()


hashtags = [
    "voorjou", "voorjoupagina", "belgium", "belgie", "belgië",
    "netherlands", "nederland", "vrtnws", "vtm", "nwsnwsnws",
    "utrecht", "tilburg", "viraal", "hln", "nl",
    "antwerp", "gent", "brussel", "grappig", "amsterdam",
    "vlaanderen", "vlaams", "nederlands", "nosnieuws"
]

target_accounts = []

url = "https://tiktok-scraper7.p.rapidapi.com/feed/search"

my_api_key = os.getenv("RAPIDAPI_KEY")

headers = {
    "x-rapidapi-host": "tiktok-scraper7.p.rapidapi.com",
    "x-rapidapi-key": my_api_key  
}

TARGET_DUTCH_COUNT = 1000
seen_video_ids = set()
total_scraped = 0
output_file = 'dutch_videos_raw.jsonl'

search_terms = hashtags + target_accounts

with open(output_file, 'a', encoding='utf-8') as f:

    for term in search_terms:
        if total_scraped >= TARGET_DUTCH_COUNT:
            break

        current_cursor = "0"
        has_more = True
        retry_count = 0

        while has_more and total_scraped < TARGET_DUTCH_COUNT:
            querystring = {
                "keywords": term,
                "region": "nl",
                "count": "30",
                "cursor": current_cursor,
                "publish_time": "0",
                "sort_type": "0"
            }

            try:
                response = requests.get(url, headers=headers, params=querystring)


                if response.status_code == 429:
                    retry_count += 1
                    if retry_count > 3:
                        print(f"Rate limit exceeded for '{term}'. Moving to next keyword...")
                        break
                    time.sleep(15)
                    continue

                elif response.status_code != 200:
                    print(f"API Error: {response.status_code} for term '{term}'")
                    break

                retry_count = 0
                data = response.json()


                videos_list = []
                meta_data = {}

                if isinstance(data, dict):
                    inner_data = data.get("data", {})


                    if isinstance(inner_data, dict):
                        videos_list = inner_data.get("videos", [])
                        meta_data = inner_data

                    elif isinstance(inner_data, list):
                        videos_list = inner_data


                    if not videos_list and isinstance(data.get("videos"), list):
                        videos_list = data.get("videos")

                if not isinstance(videos_list, list):
                    videos_list = []
                # --------------------------------

                if not videos_list:
                    print(f"No more results for: {term}")
                    break

                for video in videos_list:

                    if not isinstance(video, dict):
                        continue

                    raw_id = video.get("video_id") or video.get("aweme_id") or video.get("id")
                    video_id = str(raw_id) if raw_id is not None else ""

                    if video_id and video_id not in seen_video_ids:
                        seen_video_ids.add(video_id)


                        video['search_term_used'] = term

                        f.write(json.dumps(video, ensure_ascii=False) + '\n')
                        total_scraped += 1

                        if total_scraped >= TARGET_DUTCH_COUNT:
                            break

                if isinstance(meta_data, dict):
                    current_cursor = str(meta_data.get("cursor", int(current_cursor) + 30))
                    has_more = meta_data.get("hasMore", False) or meta_data.get("has_more", False)
                else:
                    current_cursor = str(int(current_cursor) + 30)
                    has_more = False

            except Exception as e:
                print(f"Error encountered: {e}")
                break


            time.sleep(4)



No more results for: voorjoupagina


In [2]:
print(f"Total unique videos saved: {total_scraped}")
print(f"All raw metadata saved to '{output_file}'")

Total unique videos saved: 1000
All raw metadata saved to 'dutch_videos_raw.jsonl'


In [5]:
df = pd.read_json('dutch_videos_raw.jsonl', lines=True)

In [10]:
DetectorFactory.seed = 0


try:
    df_raw = pd.read_json('dutch_videos_raw.jsonl', lines=True)

except FileNotFoundError:

    df_raw = pd.DataFrame()

if not df_raw.empty:
    cleaned_videos = []

    for index, row in df_raw.iterrows():

        text = row.get("title") or row.get("desc") or ""


        if isinstance(text, (dict, list)):
            text = str(text)
        else:
            text = str(text).strip()


        if text:
            try:
                lang = detect(text)
                if lang == 'nl':

                    raw_time = row.get("create_time")
                    formatted_date = ""
                    if raw_time and pd.notna(raw_time):
                        try:
                            formatted_date = datetime.datetime.fromtimestamp(int(raw_time)).strftime('%Y-%m-%d %H:%M:%S')
                        except (ValueError, TypeError):
                            pass


                    stats = row.get("statistics") if isinstance(row.get("statistics"), dict) else {}
                    author = row.get("author") if isinstance(row.get("author"), dict) else {}

                    video_id = row.get("video_id") or row.get("aweme_id") or row.get("id")


                    video_info = {
                        "search_hashtag": row.get("search_term_used", ""),
                        "video_id": str(video_id) if pd.notna(video_id) else "",
                        "real_video_url": f"https://www.tiktok.com/@placeholder/video/{video_id}" if pd.notna(video_id) else "",
                        "create_time": raw_time if pd.notna(raw_time) else "",
                        "publish_date": formatted_date,
                        "clean_text": text,
                        "author_name": author.get("nickname") or author.get("unique_id") or "",
                        "play_count": stats.get("play_count", 0),
                        "digg_count": stats.get("digg_count", 0)
                    }
                    cleaned_videos.append(video_info)

            except LangDetectException:

                pass


    df_cleaned = pd.DataFrame(cleaned_videos)


    df_cleaned.drop_duplicates(subset=['video_id'], keep='first', inplace=True)

    output_csv = 'final_pure_dutch_1000_for_apify.csv'
    df_cleaned.to_csv(output_csv, index=False, encoding='utf-8-sig')


    print(f"Dutch videos: {len(df_cleaned)}")
    print(f"Saved as: '{output_csv}'")


Dutch videos: 565
Saved as: 'final_pure_dutch_1000_for_apify.csv'
